# Colab-ready 教學入口：Ch15 使用 CNN 進行影像分類

本 notebook 由官方程式碼 notebook 產生，第一格加入 Colab setup，讓學生不需要手動 clone repo 或切換工作目錄。

- 官方來源：`ch15/downloading-celeba/downloading-celeba.ipynb`
- 官方 repo：https://github.com/rasbt/python-machine-learning-book-3rd-edition.git
- 書籍：Sebastian Raschka and Vahid Mirjalili, *Python Machine Learning, 3rd Ed.*, Packt Publishing, 2019
- 程式碼授權：MIT License，請參考本 repo 的 `THIRD_PARTY_NOTICES.md`

上課時請先執行下一格 setup，再依序執行原 notebook。若深度學習或大型資料章節耗時過久，請改用課堂 quick mode 或由講師示範重點 cell。


In [ ]:
# @title Colab setup for Python Machine Learning 3rd ed.
import importlib
import os
import platform
import subprocess
import sys

REPO_URL = "https://github.com/rasbt/python-machine-learning-book-3rd-edition.git"
REPO_DIR = "/content/python-machine-learning-book-3rd-edition" if os.path.exists("/content") else os.path.abspath("_python_ml_3e_official")
CHAPTER_DIR = "ch15"
EXTRA_PACKAGES = ['watermark', 'mlxtend', 'pyprind', 'tensorflow', 'tensorflow-datasets']

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

target_dir = os.path.join(REPO_DIR, CHAPTER_DIR)
os.chdir(target_dir)
print("Working directory:", os.getcwd())

def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        _run([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]:
    ensure_import(import_name, pip_name)

for pkg in EXTRA_PACKAGES:
    import_name = pkg.split("==")[0].split("[")[0].replace("-", "_")
    if pkg.startswith("tensorflow-datasets"):
        import_name = "tensorflow_datasets"
    if pkg.startswith("scikit-learn"):
        import_name = "sklearn"
    if pkg.startswith("gym=="):
        import_name = "gym"
    ensure_import(import_name, pkg)

import numpy as np

# Compatibility shims for the current Colab runtime family.
# Colab 2026.04 lists Python 3.12.13, NumPy 2.0.2, and TensorFlow 2.19.0.
# The 2019 book notebooks still use a few aliases/API shapes from older releases.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

try:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.rcParams["figure.figsize"] = (7, 5)
except Exception as exc:
    print("matplotlib setup skipped:", type(exc).__name__, exc)

try:
    import tensorflow as tf
    print("TensorFlow devices:", [device.device_type + ":" + device.name.split(":")[-1] for device in tf.config.list_physical_devices()])
except Exception as exc:
    print("TensorFlow not loaded:", type(exc).__name__, exc)

try:
    import gym

    if not getattr(gym, "_pyml3e_old_api_patch", False):
        _original_gym_make = gym.make

        class _OldStepAPIWrapper(gym.Wrapper):
            def reset(self, *args, **kwargs):
                result = self.env.reset(*args, **kwargs)
                if isinstance(result, tuple) and len(result) == 2:
                    return result[0]
                return result

            def step(self, action):
                result = self.env.step(action)
                if isinstance(result, tuple) and len(result) == 5:
                    obs, reward, terminated, truncated, info = result
                    return obs, reward, bool(terminated or truncated), info
                return result

        def _patched_make(*args, **kwargs):
            env = _original_gym_make(*args, **kwargs)
            return _OldStepAPIWrapper(env)

        gym.make = _patched_make
        gym._pyml3e_old_api_patch = True
        print("Gym old-step API wrapper enabled.")
except Exception as exc:
    print("Gym compatibility setup skipped:", type(exc).__name__, exc)

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
for mod_name in ["numpy", "pandas", "matplotlib", "sklearn", "tensorflow", "tensorflow_datasets", "gym"]:
    try:
        mod = importlib.import_module(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "installed"))
    except Exception as exc:
        print(f"{mod_name}: not loaded ({type(exc).__name__})")

print("Setup complete. Run the notebook cells below in order.")


# Fetching & Loading CelebA

In [ ]:
# ! pip install torchvision


In [ ]:
import torchvision


**Fetching CelebA dataset**

---

1. Downloading the image files manually

    - You can try setting `download=True` below. If this results in a `BadZipfile` error, we recommend downloading the `img_align_celeba.zip` file manually from http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html. In the Google Drive folder, you can find it under the `Img` folder as shown below:

In [ ]:
IPythonImage(filename='figures/gdrive-download-location-1.png', width=500)


- You can also try this direct  link: https://drive.google.com/file/d/1m8-EBPgi5MRubrm6iQjafK2QMHDBMSfJ/view?usp=sharing
- After downloading, please put this file into the `./celeba` subolder and unzip it.

2. Next,  you need to download the annotation files and put them into the same `./celeba` subfolder. The annotation files can be found under `Anno`:

In [ ]:
IPythonImage(filename='figures/gdrive-download-location-2.png', width=300)


- direct links are provided below:
  - [identity_CelebA.txt](https://drive.google.com/file/d/1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS/view?usp=sharing)
  - [list_attr_celeba.txt](https://drive.google.com/file/d/0B7EVK8r0v71pblRyaVFSWGxPY0U/view?usp=sharing&resourcekey=0-YW2qIuRcWHy_1C2VaRGL3Q)
  - [list_bbox_celeba.txt](https://drive.google.com/file/d/0B7EVK8r0v71pbThiMVRxWXZ4dU0/view?usp=sharing&resourcekey=0-z-17UMo1wt4moRL2lu9D8A)
  - [list_landmarks_align_celeba.txt](https://drive.google.com/file/d/0B7EVK8r0v71pd0FJY3Blby1HUTQ/view?usp=sharing&resourcekey=0-aFtzLN5nfdhHXpAsgYA8_g)
  - [list_landmarks_celeba.txt](https://drive.google.com/file/d/0B7EVK8r0v71pTzJIdlJWdHczRlU/view?usp=sharing&resourcekey=0-49BtYuqFDomi-1v0vNVwrQ)

In [ ]:
IPythonImage(filename='figures/gdrive-download-location-3.png', width=300)


3. Lastly, you need to download the file `list_eval_partition.txt` and place it under `./celeba`:

- [list_eval_partition.txt](https://drive.google.com/file/d/0B7EVK8r0v71pY0NSMzRuSXJEVkk/view?usp=sharing&resourcekey=0-i4TGCi_51OtQ5K9FSp4EDg)

After completing steps 1-3 above, please ensure you have the following files in your `./celeba` subfolder, and the files are non-empty (that is, they have similar file sizes as shown below):

In [ ]:
IPythonImage(filename='figures/celeba-files.png', width=400)


---

In [ ]:
image_path = './'
celeba_dataset = torchvision.datasets.CelebA(image_path, split='train', target_type='attr', download=False)

assert isinstance(celeba_dataset, torch.utils.data.Dataset)


In [ ]:
example = next(iter(celeba_dataset))
print(example)


In [ ]:
from itertools import islice
fig = plt.figure(figsize=(12, 8))
for i, (image, attributes) in islice(enumerate(celeba_dataset), 18):
    ax = fig.add_subplot(3, 6, i+1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(image)
    ax.set_title(f'{attributes[31]}', size=15)
    
#plt.savefig('figures/12_05.pdf')
plt.show()
